GPU required (Settings -> Accelerator -> GPU T4 x2, Internet -> On).

In [ ]:
CSV_PATH = "/kaggle/input/datasets/pratiktimilsina/nepal-tourism-reviews/nepal_tourism_reviews.csv"

## 1. Install & import

In [ ]:
!pip install -q vaderSentiment==3.3.2 statsmodels==0.15.0
try:
    import transformers, datasets, accelerate
except ImportError:
    !pip install -q transformers datasets accelerate

import re
import json
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                              accuracy_score, cohen_kappa_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from datasets import Dataset
import transformers
import sklearn
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)

import nltk
for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag, word_tokenize, sent_tokenize

import os
os.makedirs("/kaggle/working/models", exist_ok=True)
os.makedirs("/kaggle/working/outputs/metrics", exist_ok=True)
os.makedirs("/kaggle/working/outputs/figures", exist_ok=True)
os.makedirs("/kaggle/working/outputs/rankings", exist_ok=True)

RANDOM_STATE = 42
LABELS = ["negative", "neutral", "positive"]
plt.rcParams["figure.dpi"] = 110

import random
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f"transformers={transformers.__version__}  torch={torch.__version__}  sklearn={sklearn.__version__}")

## 2. Load raw data

In [ ]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
print(df["label"].value_counts())
print(df.groupby(["city", "domain"]).size())
df.head(3)

## 3. Clean text

In [ ]:
URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_RE = re.compile(r"<.*?>")
REPEAT_CHAR_RE = re.compile(r"(.)\1{2,}")
SPECIAL_RE = re.compile(r"[^a-z0-9\s.,!?']")
MULTISPACE_RE = re.compile(r"\s+")

_lemmatizer = WordNetLemmatizer()

def _wordnet_pos(tag):
    if tag.startswith("J"): return wordnet.ADJ
    if tag.startswith("V"): return wordnet.VERB
    if tag.startswith("N"): return wordnet.NOUN
    if tag.startswith("R"): return wordnet.ADV
    return wordnet.NOUN

def light_clean(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = REPEAT_CHAR_RE.sub(r"\1\1", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

def deep_clean(text):
    text = light_clean(text).lower()
    text = SPECIAL_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    if not text:
        return ""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    lemmas = [_lemmatizer.lemmatize(tok, _wordnet_pos(tag)) for tok, tag in tagged]
    return " ".join(lemmas)

df["sentence_review"] = df["review"].apply(light_clean)
df["clean_review"] = df["review"].apply(deep_clean)

before = len(df)
df = df[df["clean_review"].str.len() > 0].reset_index(drop=True)
print(f"dropped {before - len(df)} rows with empty text after cleaning; {len(df)} remain")

## 4. Train/validation/test split

In [ ]:
df["place_id"] = df["city"].astype(str) + "||" + df["domain"].astype(str) + "||" + df["place"].astype(str)

def stratified_group_split(df, test_size=0.2, random_state=RANDOM_STATE):
    train_parts, test_parts = [], []
    for (city, domain), sub in df.groupby(["city", "domain"]):
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        tr_idx, te_idx = next(gss.split(sub, groups=sub["place_id"]))
        train_parts.append(sub.iloc[tr_idx])
        test_parts.append(sub.iloc[te_idx])
        print(f"{city}/{domain}: {sub['place_id'].nunique()} places -> "
              f"{sub.iloc[tr_idx]['place_id'].nunique()} train / {sub.iloc[te_idx]['place_id'].nunique()} test")
    train = pd.concat(train_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    test = pd.concat(test_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    assert not (set(train["place_id"]) & set(test["place_id"])), "place leakage!"
    return train, test

train, test = stratified_group_split(df, test_size=0.2, random_state=RANDOM_STATE)
train_fit, val = stratified_group_split(train, test_size=0.2, random_state=RANDOM_STATE)
assert not (set(train_fit["place_id"]) & set(val["place_id"])), "place leakage: train_fit/val"
assert not (set(train_fit["place_id"]) & set(test["place_id"])), "place leakage: train_fit/test"
assert not (set(val["place_id"]) & set(test["place_id"])), "place leakage: val/test"

le = LabelEncoder().fit(train_fit["label"])
y_train, y_val, y_test = le.transform(train_fit["label"]), le.transform(val["label"]), le.transform(test["label"])
print(f"\ntrain_fit={len(train_fit)}  val={len(val)}  test={len(test)}  classes={list(le.classes_)}")

split_distribution = pd.concat([
    train_fit.assign(split="train_fit"), val.assign(split="val"), test.assign(split="test")
]).groupby(["split", "label"]).size().unstack(fill_value=0)
split_pct = split_distribution.div(split_distribution.sum(axis=1), axis=0)
print("\nlabel distribution by split (counts):")
print(split_distribution)
print("\nlabel distribution by split (fraction):")
print(split_pct.round(3))

split_assignments = pd.concat([
    train_fit.assign(split="train_fit"), val.assign(split="val"), test.assign(split="test")
])[["city", "domain", "place", "place_id", "review", "split"]]
split_assignments.to_csv("/kaggle/working/outputs/metrics/split_assignments.csv", index=False)
print(f"\nsaved split_assignments.csv ({len(split_assignments)} rows)")

## 5. Train TF-IDF + Logistic Regression

In [ ]:
def build_pipeline(max_features, ngram_range, min_df, max_df, C, class_weight):
    return Pipeline([
        ("tfidf", TfidfVectorizer(max_features=max_features, ngram_range=ngram_range,
                                   min_df=min_df, max_df=max_df, sublinear_tf=True)),
        ("clf", LogisticRegression(C=C, max_iter=2000, class_weight=class_weight,
                                    random_state=RANDOM_STATE)),
    ])

X_train_text = train_fit["clean_review"]
X_val_text = val["clean_review"]
X_test_text = test["clean_review"]

baseline = build_pipeline(max_features=20000, ngram_range=(1, 2), min_df=3, max_df=0.98,
                           C=1.0, class_weight="balanced")
baseline.fit(X_train_text, y_train)
print("=== Baseline (val) ===")
print(classification_report(y_val, baseline.predict(X_val_text), target_names=le.classes_, zero_division=0))

candidates = []
for C in [0.5, 1.0, 3.0, 5.0]:
    for ngram_range in [(1, 2), (1, 3)]:
        for class_weight in ["balanced", None]:
            pipe = build_pipeline(max_features=30000, ngram_range=ngram_range, min_df=2, max_df=0.95,
                                   C=C, class_weight=class_weight)
            pipe.fit(X_train_text, y_train)
            mf1 = f1_score(y_val, pipe.predict(X_val_text), average="macro")
            candidates.append((mf1, C, ngram_range, class_weight, pipe))
            print(f"C={C} ngram={ngram_range} cw={class_weight} -> val macro-F1={mf1:.4f}")

candidates.sort(key=lambda x: -x[0])
best_mf1, best_C, best_ngram, best_cw, _ = candidates[0]
print(f"\nBest config (selected on val): C={best_C} ngram={best_ngram} class_weight={best_cw} val macro-F1={best_mf1:.4f}")

X_trainval_text = pd.concat([X_train_text, X_val_text])
y_trainval = np.concatenate([y_train, y_val])
tfidf_lr = build_pipeline(max_features=30000, ngram_range=best_ngram, min_df=2, max_df=0.95,
                           C=best_C, class_weight=best_cw)
tfidf_lr.fit(X_trainval_text, y_trainval)

tfidf_preds = tfidf_lr.predict(X_test_text)
print("\n=== Final test evaluation ===")
print(classification_report(y_test, tfidf_preds, target_names=le.classes_, zero_division=0))

joblib.dump(tfidf_lr, "/kaggle/working/models/tfidf_lr_pipeline.joblib")
joblib.dump(le, "/kaggle/working/models/label_encoder.joblib")

## 6. Fine-tune two transformers, with class-weighted loss

In [ ]:
class_counts = np.bincount(y_train, minlength=len(le.classes_))
class_weights = len(y_train) / (len(le.classes_) * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("class weights (by label index, matching le.classes_ =", list(le.classes_), "):", class_weights)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, num_labels=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.num_labels = num_labels

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean(), "macro_f1": f1_score(labels, preds, average="macro")}

def finetune_transformer(model_name, save_subdir, epochs=4):
    tok = AutoTokenizer.from_pretrained(model_name)

    def _tok(batch):
        return tok(batch["text"], truncation=True, max_length=256)

    tr_ds = Dataset.from_pandas(
        train_fit[["sentence_review"]].assign(labels=y_train).rename(columns={"sentence_review": "text"})
    ).map(_tok, batched=True)
    val_ds = Dataset.from_pandas(
        val[["sentence_review"]].assign(labels=y_val).rename(columns={"sentence_review": "text"})
    ).map(_tok, batched=True)
    te_ds = Dataset.from_pandas(
        test[["sentence_review"]].assign(labels=y_test).rename(columns={"sentence_review": "text"})
    ).map(_tok, batched=True)

    mdl = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(le.classes_))
    coll = DataCollatorWithPadding(tokenizer=tok)
    args = TrainingArguments(
        output_dir=f"/kaggle/working/checkpoints_{save_subdir}",
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
        num_train_epochs=epochs, weight_decay=0.01,
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        logging_steps=50, report_to="none",
        seed=RANDOM_STATE, data_seed=RANDOM_STATE,
    )
    trn = WeightedTrainer(model=mdl, args=args, train_dataset=tr_ds, eval_dataset=val_ds,
                           data_collator=coll, compute_metrics=compute_metrics,
                           class_weights=class_weights_tensor, num_labels=len(le.classes_))
    trn.train()

    preds_logits = trn.predict(te_ds)
    preds = np.argmax(preds_logits.predictions, axis=-1)
    print(f"\n=== {model_name} (final test evaluation) ===")
    print(classification_report(y_test, preds, target_names=le.classes_, zero_division=0))

    trn.save_model(f"/kaggle/working/models/{save_subdir}")
    tok.save_pretrained(f"/kaggle/working/models/{save_subdir}")
    return trn, mdl, tok, preds

### 6a. `distilbert-base-multilingual-cased`

In [ ]:
DISTIL_MODEL_NAME = "distilbert-base-multilingual-cased"
distil_trainer, distil_model, distil_tokenizer, distil_preds = finetune_transformer(
    DISTIL_MODEL_NAME, "distilbert_sentiment")

### 6b. `bert-base-uncased` (English-only)

In [ ]:
EN_MODEL_NAME = "bert-base-uncased"
bert_trainer, bert_model, bert_tokenizer, bert_preds = finetune_transformer(
    EN_MODEL_NAME, "bert_en_sentiment")

## 7. Five-way benchmark

In [ ]:
test = test.copy()
test["tfidf_lr_pred"] = le.inverse_transform(tfidf_preds)
test["distilbert_pred"] = le.inverse_transform(distil_preds)
test["bert_en_pred"] = le.inverse_transform(bert_preds)

predictors = {"tfidf_lr": "tfidf_lr_pred", "distilbert": "distilbert_pred", "bert_en": "bert_en_pred",
              "llama_3.1_8b": "llama_label", "qwen2.5_14b": "qwen_label"}

place_arr = test["place_id"].to_numpy()
unique_test_places = np.unique(place_arr)
place_to_idx = {p: np.where(place_arr == p)[0] for p in unique_test_places}

def bootstrap_metric(y_true, y_pred, metric_fn, n_boot=2000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n_places = len(unique_test_places)
    values = []
    for _ in range(n_boot):
        sampled_places = rng.choice(unique_test_places, size=n_places, replace=True)
        idx = np.concatenate([place_to_idx[p] for p in sampled_places])
        values.append(metric_fn(y_true[idx], y_pred[idx]))
    return float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))

y_true_arr = test["label"].to_numpy()
results = {}
for name, col in predictors.items():
    y_pred_arr = test[col].to_numpy()
    acc = accuracy_score(y_true_arr, y_pred_arr)
    mf1 = f1_score(y_true_arr, y_pred_arr, average="macro", labels=LABELS)
    acc_ci = bootstrap_metric(y_true_arr, y_pred_arr, accuracy_score)
    mf1_ci = bootstrap_metric(y_true_arr, y_pred_arr,
                               lambda a, b: f1_score(a, b, average="macro", labels=LABELS))
    report = classification_report(y_true_arr, y_pred_arr, labels=LABELS,
                                    output_dict=True, zero_division=0)
    results[name] = {"accuracy": acc, "accuracy_ci95": acc_ci,
                      "macro_f1": mf1, "macro_f1_ci95": mf1_ci, "report": report}
    print(f"\n=== {name} ===  accuracy={acc:.4f} (place-cluster 95% CI {acc_ci[0]:.4f}-{acc_ci[1]:.4f})  "
          f"macro-F1={mf1:.4f} (place-cluster 95% CI {mf1_ci[0]:.4f}-{mf1_ci[1]:.4f})")
    print(f"  per-class F1 -- negative={report['negative']['f1-score']:.4f}  "
          f"neutral={report['neutral']['f1-score']:.4f}  positive={report['positive']['f1-score']:.4f}")
    print(classification_report(y_true_arr, y_pred_arr, labels=LABELS, zero_division=0))

agreement = {}
cols = list(predictors.values())
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        a, b = cols[i], cols[j]
        kappa = cohen_kappa_score(test[a], test[b])
        agreement[f"{a}_vs_{b}"] = kappa
        print(f"kappa({a}, {b}) = {kappa:.4f}")

breakdown = (test.assign(correct=lambda d: d["label"] == d["tfidf_lr_pred"])
             .groupby(["city", "domain"])["correct"].mean().reset_index()
             .rename(columns={"correct": "tfidf_lr_accuracy"}))
print("\nTF-IDF+LR accuracy by city/domain:")
print(breakdown.to_string(index=False))

all_wrong_cols = list(predictors.values())
all_wrong_mask = np.all([test["label"] != test[c] for c in all_wrong_cols], axis=0)
all_wrong = test[all_wrong_mask]
print(f"\nAll 5 models disagree with reference: {len(all_wrong)}/{len(test)} "
      f"({100*len(all_wrong)/len(test):.1f}%)")

with open("/kaggle/working/outputs/metrics/five_way_benchmark.json", "w") as f:
    json.dump({"results": results, "agreement": agreement,
               "breakdown_by_city_domain": breakdown.to_dict(orient="records"),
               "n_all_wrong": int(len(all_wrong)), "n_test": int(len(test))},
              f, indent=2, default=str)
test.to_csv("/kaggle/working/outputs/metrics/test_predictions.csv", index=False)
all_wrong.to_csv("/kaggle/working/outputs/metrics/hard_cases_all_wrong.csv", index=False)

### 7a. Statistical significance (place-cluster bootstrap, Holm-corrected)

In [ ]:
import itertools
from statsmodels.stats.multitest import multipletests

def bootstrap_diff(pred_a, pred_b, metric_fn, n_boot=2000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n_places = len(unique_test_places)
    diffs = []
    for _ in range(n_boot):
        sampled_places = rng.choice(unique_test_places, size=n_places, replace=True)
        idx = np.concatenate([place_to_idx[p] for p in sampled_places])
        diffs.append(metric_fn(y_true_arr[idx], pred_a[idx]) - metric_fn(y_true_arr[idx], pred_b[idx]))
    return np.array(diffs)

def cluster_permutation_p(pred_a, pred_b, metric_fn):
    observed = metric_fn(y_true_arr, pred_a) - metric_fn(y_true_arr, pred_b)
    null_diffs = []
    for swap_pattern in itertools.product([False, True], repeat=len(unique_test_places)):
        perm_a = np.empty(len(y_true_arr), dtype=pred_a.dtype)
        perm_b = np.empty(len(y_true_arr), dtype=pred_b.dtype)
        for place, swap in zip(unique_test_places, swap_pattern):
            idx = place_to_idx[place]
            if swap:
                perm_a[idx] = pred_b[idx]
                perm_b[idx] = pred_a[idx]
            else:
                perm_a[idx] = pred_a[idx]
                perm_b[idx] = pred_b[idx]
        null_diffs.append(metric_fn(y_true_arr, perm_a) - metric_fn(y_true_arr, perm_b))
    null_diffs = np.array(null_diffs)
    return float(observed), float((np.abs(null_diffs) >= abs(observed) - 1e-12).mean())

names_list = list(predictors.keys())
cluster_results = []
for i in range(len(names_list)):
    for j in range(i + 1, len(names_list)):
        a, b_name = names_list[i], names_list[j]
        pred_a = test[predictors[a]].to_numpy()
        pred_b = test[predictors[b_name]].to_numpy()
        diffs = bootstrap_diff(pred_a, pred_b, accuracy_score)
        observed, p = cluster_permutation_p(pred_a, pred_b, accuracy_score)
        cluster_results.append({
            "model_a": a, "model_b": b_name,
            "accuracy_diff": observed,
            "accuracy_diff_ci95": [float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))],
            "p_value": p,
        })

pvals = [r["p_value"] for r in cluster_results]
reject, p_adj, _, _ = multipletests(pvals, alpha=0.05, method="holm")
for r, adj, sig in zip(cluster_results, p_adj, reject):
    r["p_value_holm"] = float(adj)
    r["significant_p05"] = bool(sig)

print(f"{'model A':<12} {'model B':<12} {'acc diff':>9} {'CI low':>8} {'CI high':>8} {'p-raw':>8} {'p-holm':>8}  significant?")
for r in cluster_results:
    sig = "yes (p<0.05)" if r["significant_p05"] else "no"
    print(f"{r['model_a']:<12} {r['model_b']:<12} {r['accuracy_diff']:>9.4f} "
          f"{r['accuracy_diff_ci95'][0]:>8.4f} {r['accuracy_diff_ci95'][1]:>8.4f} "
          f"{r['p_value']:>8.4f} {r['p_value_holm']:>8.4f}  {sig}")

with open("/kaggle/working/outputs/metrics/cluster_significance.json", "w") as f:
    json.dump(cluster_results, f, indent=2, default=str)

### 7b. Cases where all 5 models disagree with the reference label

In [ ]:
print(f"{len(all_wrong)} reviews where all 5 models disagree with the reference label\n")

print("Reference-label distribution of these cases (vs. overall test set):")
print(pd.DataFrame({
    "all_disagree_pct": all_wrong["label"].value_counts(normalize=True).round(3),
    "overall_test_pct": test["label"].value_counts(normalize=True).round(3),
}))

print("\nStar rating distribution of these cases:")
print(all_wrong["rating"].value_counts().sort_index())

disagree_wc = all_wrong["review"].str.split().str.len()
overall_wc = test["review"].str.split().str.len()
print(f"\nReview length (words) -- these cases: median={disagree_wc.median():.0f}, "
      f"mean={disagree_wc.mean():.1f} | overall test set: median={overall_wc.median():.0f}, "
      f"mean={overall_wc.mean():.1f}")

print("\n10 examples (reference label -> what each model predicted):")
sample_cols = ["label", "rating", "tfidf_lr_pred", "distilbert_pred", "bert_en_pred",
               "llama_label", "qwen_label", "review"]
with pd.option_context("display.max_colwidth", 100):
    print(all_wrong[sample_cols].sample(min(10, len(all_wrong)), random_state=1).to_string(index=False))

## 8. Post-hoc exploratory aspect analysis — scored 4 ways

In [ ]:
HOTEL_ASPECTS = {
    "room": ["room", "rooms", "bedroom", "suite", "furnishing", "interior"],
    "bathroom": ["bathroom", "bathrooms", "toilet", "shower", "washroom", "restroom", "hot water", "geyser"],
    "cleanliness": ["clean", "cleanliness", "hygien", "dust", "dirty", "spotless", "tidy", "smell", "stain"],
    "staff": ["staff", "employee", "employees", "receptionist", "front desk", "manager", "waiter",
              "service", "helpful", "friendly", "rude", "hospitality", "host"],
    "food": ["food", "breakfast", "buffet", "restaurant", "dinner", "lunch", "meal", "cuisine",
             "menu", "taste", "delicious"],
    "wifi": ["wifi", "wi-fi", "internet", "connection", "network"],
    "location": ["location", "located", "nearby", "walking distance", "central", "surrounding"],
    "comfort": ["comfort", "comfortable", "cozy", "bed", "pillow", "mattress", "quiet room"],
    "amenities": ["amenities", "facility", "facilities", "pool", "gym", "spa", "parking", "elevator", "lift"],
    "noise": ["noise", "noisy", "loud", "quiet", "silence"],
    "price": ["price", "expensive", "cheap", "cost", "value for money", "affordable", "overpriced", "rate"],
    "checkin_checkout": ["check-in", "checkin", "check in", "checkout", "check-out", "check out"],
    "transport_access": ["transport", "taxi", "shuttle", "airport", "pickup", "drop", "commute", "access"],
    "view": ["view", "scenery", "overlook", "balcony view", "mountain view", "lake view"],
}
ATTRACTION_ASPECTS = {
    "view": ["view", "scenery", "scenic", "panorama", "landscape", "overlook", "vista"],
    "access": ["access", "accessible", "entrance", "entry", "gate", "path", "trail", "walk", "steps", "stairs"],
    "safety": ["safety", "safe", "danger", "dangerous", "risky", "secure", "unsafe"],
    "maintenance": ["maintain", "maintenance", "well kept", "run down", "renovat", "broken", "worn"],
    "boat": ["boat", "boating", "canoe", "rowing", "paddle"],
    "temple": ["temple", "temples", "stupa", "shrine", "monastery", "pagoda", "worship", "monk"],
    "waterfall": ["waterfall", "falls", "cascade", "fall"],
    "activities": ["activity", "activities", "adventure", "trek", "hiking", "zipline", "rafting", "trekking"],
    "paragliding": ["paraglid", "paragliding", "tandem flight", "takeoff", "landing"],
    "environment": ["environment", "nature", "pollution", "clean air", "greenery", "peaceful", "crowd", "crowded"],
    "shop": ["shop", "shops", "shopping", "market", "markets", "souvenir", "stall", "stalls", "vendor"],
    "guides": ["guide", "guides", "guiding", "instructor", "tour guide"],
    "food_stalls": ["food stall", "street food", "snack", "cafe", "eatery"],
    "transport": ["transport", "taxi", "bus", "cable car", "ropeway", "parking"],
    "lake": ["lake", "lakeside", "lakefront", "lakeshore"],
    "cave": ["cave", "caves", "cavern", "stalactite", "stalagmite"],
    "museum": ["museum", "museums", "gallery", "exhibit", "exhibition"],
    "earthquake_damage": ["earthquake", "quake damage"],
}

def compile_patterns(aspect_dict):
    return {a: re.compile(r"\b(" + "|".join(re.escape(k) for k in kws) + r")\b", re.IGNORECASE)
            for a, kws in aspect_dict.items()}

HOTEL_PATTERNS = compile_patterns(HOTEL_ASPECTS)
ATTRACTION_PATTERNS = compile_patterns(ATTRACTION_ASPECTS)
def get_patterns(domain):
    return HOTEL_PATTERNS if domain == "Hotel" else ATTRACTION_PATTERNS

places_by_domain = {d: sub["place"].unique().tolist() for d, sub in df.groupby("domain")}
def other_places_pattern(domain, own_place):
    others = [p for p in places_by_domain[domain] if p != own_place]
    if not others:
        return None
    return re.compile(r"\b(" + "|".join(re.escape(p) for p in others) + r")\b", re.IGNORECASE)

CLAUSE_SPLIT_RE = re.compile(r"\b(but|however|although|though|yet|while|whereas)\b", re.IGNORECASE)
def split_clauses(sentence):
    parts = CLAUSE_SPLIT_RE.split(sentence)
    clauses = [p.strip() for i, p in enumerate(parts) if i % 2 == 0 and p.strip()]
    return clauses if len(clauses) > 1 else [sentence]

rows, all_clauses = [], set()
n_with_match = 0
n_with_match_post_filter = 0
n_dropped_other_place = 0
for review_id, row in df.iterrows():
    sentences = sent_tokenize(row["sentence_review"])
    if not sentences:
        continue
    clauses_in_review = [cl for s in sentences for cl in split_clauses(s)]
    patterns = get_patterns(row["domain"])
    other_pat = other_places_pattern(row["domain"], row["place"])
    pairs = [(aspect, cl) for cl in clauses_in_review for aspect, pat in patterns.items() if pat.search(cl)]
    if not pairs:
        continue
    n_with_match += 1
    review_survived = False
    for aspect, cl in pairs:
        if other_pat is not None and other_pat.search(cl):
            n_dropped_other_place += 1
            continue
        review_survived = True
        all_clauses.add(cl)
        rows.append({"city": row["city"], "domain": row["domain"], "place": row["place"],
                      "aspect": aspect, "clause": cl, "review_id": review_id})
    if review_survived:
        n_with_match_post_filter += 1

unique_sentences = list(all_clauses)
coverage_raw = n_with_match / len(df)
coverage = n_with_match_post_filter / len(df)
print(f"aspect coverage, raw (before place filter): {n_with_match}/{len(df)} reviews ({coverage_raw:.1%})")
print(f"aspect coverage, post-filter (what's actually usable): {n_with_match_post_filter}/{len(df)} reviews ({coverage:.1%})")
print(f"dropped {n_dropped_other_place} aspect-clause pairs that name a different place "
      f"({n_with_match - n_with_match_post_filter} reviews lost ALL their matches to this filter)")
print(f"{len(unique_sentences)} unique aspect-matching clauses to score")

In [ ]:
classes = list(le.classes_)
pos_idx, neg_idx = classes.index("positive"), classes.index("negative")

cleaned_for_tfidf = [deep_clean(s) for s in unique_sentences]
tfidf_proba = tfidf_lr.predict_proba(cleaned_for_tfidf)
tfidf_scores = dict(zip(unique_sentences, tfidf_proba[:, pos_idx] - tfidf_proba[:, neg_idx]))

vader = SentimentIntensityAnalyzer()
vader_scores = {s: vader.polarity_scores(s)["compound"] for s in unique_sentences}

def score_batch_transformer(model, tokenizer, sentences, batch_size=64):
    model.eval()
    device = model.device
    scores = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i + batch_size]
            enc = tokenizer(batch, truncation=True, padding=True, max_length=256,
                             return_tensors="pt").to(device)
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            scores.extend((probs[:, pos_idx] - probs[:, neg_idx]).tolist())
    return scores

distilbert_scores = dict(zip(unique_sentences,
                              score_batch_transformer(distil_model, distil_tokenizer, unique_sentences)))
bert_en_scores = dict(zip(unique_sentences,
                           score_batch_transformer(bert_model, bert_tokenizer, unique_sentences)))

aspect_df = pd.DataFrame(rows)
aspect_df["tfidf_lr_score"] = aspect_df["clause"].map(tfidf_scores)
aspect_df["vader_score"] = aspect_df["clause"].map(vader_scores)
aspect_df["distilbert_score"] = aspect_df["clause"].map(distilbert_scores)
aspect_df["bert_en_score"] = aspect_df["clause"].map(bert_en_scores)
aspect_df.to_csv("/kaggle/working/outputs/rankings/aspect_clauses.csv", index=False)

corr = aspect_df[["tfidf_lr_score", "distilbert_score", "bert_en_score", "vader_score"]].corr()
print("Correlation between the four scoring methods:")
print(corr.round(3))

In [ ]:
review_level = (aspect_df.groupby(["city", "domain", "place", "aspect", "review_id"])
                [["tfidf_lr_score", "distilbert_score", "bert_en_score", "vader_score"]]
                .mean().reset_index())

agg = (review_level.groupby(["city", "domain", "place", "aspect"])
       .agg(mean_tfidf_score=("tfidf_lr_score", "mean"),
            mean_distilbert_score=("distilbert_score", "mean"),
            mean_bert_en_score=("bert_en_score", "mean"),
            mean_vader_score=("vader_score", "mean"),
            unique_reviews=("review_id", "count"))
       .reset_index())
clause_counts = (aspect_df.groupby(["city", "domain", "place", "aspect"])
                  .size().reset_index(name="aspect_count"))
agg = agg.merge(clause_counts, on=["city", "domain", "place", "aspect"])
agg.to_csv("/kaggle/working/outputs/rankings/aspect_aggregate.csv", index=False)

overall = (df.groupby(["city", "domain", "place"])
           .agg(mean_rating=("rating", "mean"), n_reviews=("rating", "count"))
           .reset_index().sort_values(["domain", "mean_rating"], ascending=[True, False]))
overall.to_csv("/kaggle/working/outputs/rankings/overall_place_ranking.csv", index=False)

top3 = (agg[agg["unique_reviews"] >= 5].sort_values("mean_tfidf_score", ascending=False)
        .groupby(["domain", "aspect"]).head(3)).copy()
top3["interpretation"] = "directional"
top3.to_csv("/kaggle/working/outputs/rankings/aspect_directional_rankings.csv", index=False)

with open("/kaggle/working/outputs/metrics/aspect_pipeline_summary.json", "w") as f:
    json.dump({"n_reviews": int(len(df)), "aspect_coverage": coverage,
               "aspect_coverage_raw": coverage_raw,
               "n_aspect_clause_pairs": int(len(aspect_df)),
               "n_dropped_other_place_mentions": int(n_dropped_other_place),
               "scorer_correlations": corr.round(3).to_dict()}, f, indent=2)

print(top3.head(10).to_string(index=False))

## 9. Figures

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(23, 4.5))
plot_cols = [("tfidf_lr_pred", "TF-IDF + LR"), ("distilbert_pred", "DistilBERT\n(multilingual)"),
             ("bert_en_pred", "BERT\n(English-only)"), ("llama_label", "Llama-3.1-8B"),
             ("qwen_label", "Qwen2.5-14B")]
for ax, (col, name) in zip(axes, plot_cols):
    cm = confusion_matrix(test["label"], test[col], labels=LABELS, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", cbar=False,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax, vmin=0, vmax=1)
    ax.set_title(name); ax.set_xlabel("Predicted")
    ax.set_ylabel("Reference label" if ax is axes[0] else "")
fig.suptitle("Row-normalized confusion matrices vs. reference label", y=1.03)
fig.tight_layout()
fig.savefig("/kaggle/working/outputs/figures/confusion_matrices.png", bbox_inches="tight")
plt.show()

In [ ]:
names = {"tfidf_lr": "TF-IDF+LR", "distilbert": "DistilBERT\n(multilingual)",
         "bert_en": "BERT\n(English-only)", "llama_3.1_8b": "Llama-3.1-8B\n(zero-shot)",
         "qwen2.5_14b": "Qwen2.5-14B\n(zero-shot)"}
accs = [results[k]["accuracy"] for k in names]
f1s = [results[k]["macro_f1"] for k in names]
x = np.arange(len(names)); width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width/2, accs, width, label="Accuracy", color="#2b6cb0")
ax.bar(x + width/2, f1s, width, label="Macro-F1", color="#dd6b20")
ax.set_xticks(x); ax.set_xticklabels(names.values()); ax.set_ylim(0, 1)
ax.set_title("Test-set sentiment performance: fine-tuned models vs. zero-shot LLM baselines")
ax.legend()
for i, (a, fv) in enumerate(zip(accs, f1s)):
    ax.text(i - width/2, a + 0.02, f"{a:.3f}", ha="center", fontsize=9)
    ax.text(i + width/2, fv + 0.02, f"{fv:.3f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig("/kaggle/working/outputs/figures/model_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
def plot_aspect_ranking(city, domain, aspect, score_col="mean_tfidf_score", top_n=10, show=True):
    sub = agg[(agg["city"] == city) & (agg["domain"] == domain) & (agg["aspect"] == aspect) & (agg["unique_reviews"] >= 5)]
    sub = sub.sort_values(score_col, ascending=True).tail(top_n)
    if sub.empty:
        print(f"no data for {city}/{domain}/{aspect} (fewer than 5 unique reviews for every place)")
        return None
    fig, ax = plt.subplots(figsize=(7, max(2.5, 0.4 * len(sub))))
    colors = ["#c53030" if v < 0 else "#2b6cb0" for v in sub[score_col]]
    ax.barh(sub["place"], sub[score_col], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel(f"mean sentence-level score ({score_col})")
    ax.set_title(f"{city} {domain} ranking — aspect: {aspect}")
    fig.tight_layout()
    fname = f"/kaggle/working/outputs/figures/aspect_{city.lower()}_{domain.lower()}_{aspect}.png"
    fig.savefig(fname, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fname

all_combos = agg[["city", "domain", "aspect"]].drop_duplicates().sort_values(["city", "domain", "aspect"]).values.tolist()
print(f"generating up to {len(all_combos)} aspect ranking charts...")

saved, skipped = [], []
for city, domain, aspect in all_combos:
    fname = plot_aspect_ranking(city, domain, aspect, show=True)
    (saved if fname else skipped).append((city, domain, aspect))

print(f"\nsaved {len(saved)} charts to outputs/figures/")
if skipped:
    print(f"skipped {len(skipped)} (fewer than 5 unique reviews for every place in that aspect): {skipped}")

In [ ]:
import subprocess, sys
with open("/kaggle/working/outputs/metrics/environment.txt", "w") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, text=True)
print("saved environment.txt")